# SurahChain Pre-train على Kaggle (ليلي / بدون مراقبة)

**قبل التشغيل:**
1. Settings → Accelerator → **GPU T4** (أو T4×2 إن وُجد)
2. Settings → Internet → **ON**
3. Add-ons → Secrets → أضف `GITHUB_TOKEN` (صلاحية repo)
4. **Save Version → Save & Run All** ثم يمكنك إغلاق المتصفح والنوم

القواعد:
- سلسلة السور **114 كما هي** (لا دمج / لا تغيير أبعاد)
- التقوية: `d_model=256` + انتباه أقوى (QK-Norm + Gated Attention)
- تسريع: batch أكبر + `torch.compile` إن أمكن + حفظ كل عصر

In [ ]:
# ===== إعدادات ليلية =====
SCN_PRESET = "medium"   # d=256, pre/post=4, سلسلة 114 كما هي
SCN_N = 60000
SCN_EPOCHS = 30         # حقب هذه الجولة (مع resume لاحقاً)
SCN_BATCH = 24          # ارفع إلى 32 إن لم تنفد الذاكرة
SCN_FRESH = True        # True لأول تشغيل medium فقط
SCN_COMPILE = True      # تسريع على GPU
SCN_QK_NORM = True
SCN_GATED_ATTN = True
AUTO_PUSH = True
REPO = "aliahmed369000000-ai/Neural-Service-Mesh"
BRANCH = "main"
print("ready", SCN_PRESET, "N", SCN_N, "epochs", SCN_EPOCHS)

In [ ]:
import os, torch
print("cuda:", torch.cuda.is_available(), "gpus:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))
!pip -q install datasets huggingface_hub

In [ ]:
# التوكن من Kaggle Secrets (آمن أكثر من لصقه في الخلية)
try:
    from kaggle_secrets import UserSecretsClient
    GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
    print("token: from Kaggle Secrets")
except Exception as e:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
    print("token fallback:", "set" if GITHUB_TOKEN else "MISSING", e)

assert GITHUB_TOKEN, "أضف Secret اسمه GITHUB_TOKEN في Kaggle"

import pathlib
work = pathlib.Path("/kaggle/working/Neural-Service-Mesh")
url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"
if work.exists():
    %cd /kaggle/working/Neural-Service-Mesh
    !git remote set-url origin {url}
    !git pull origin {BRANCH}
else:
    %cd /kaggle/working
    !git clone --depth 1 -b {BRANCH} {url} Neural-Service-Mesh
    %cd /kaggle/working/Neural-Service-Mesh

!git config user.email "nsm-bot@users.noreply.github.com"
!git config user.name "NSM Bot"
print("cwd", os.getcwd())

In [ ]:
import os
os.environ["SCN_N"] = str(SCN_N)
!python experiments/surah_chain_network/prepare_pretrain_data.py

In [ ]:
import os
os.environ["SCN_PRESET"] = SCN_PRESET
os.environ["SCN_N"] = str(SCN_N)
os.environ["SCN_EPOCHS"] = str(SCN_EPOCHS)
os.environ["SCN_BATCH"] = str(SCN_BATCH)
os.environ["SCN_FRESH"] = "1" if SCN_FRESH else "0"
os.environ["SCN_COMPILE"] = "1" if SCN_COMPILE else "0"
os.environ["SCN_QK_NORM"] = "1" if SCN_QK_NORM else "0"
os.environ["SCN_GATED_ATTN"] = "1" if SCN_GATED_ATTN else "0"
os.environ["SCN_CHAIN_SCALE"] = "1"  # سلسلة 114 دون تغيير أبعاد

!python experiments/surah_chain_network/train_pretrain_torch.py

In [ ]:
from pathlib import Path
import subprocess

if not AUTO_PUSH:
    print("skip push")
else:
    exp = Path("experiments/surah_chain_network")
    ckpt = exp / "checkpoints"
    files = list(ckpt.glob("*.pt")) + list(ckpt.glob("pretrain_state_*.json")) + list(ckpt.glob("pretrain_torch_state.json"))
    files += list(exp.glob("tokenizer_vocab_pretrain*.json"))
    for f in files:
        if f.exists() and f.stat().st_size > 100:
            print("add", f, f.stat().st_size)
            subprocess.run(["git", "add", "-f", str(f)], check=False)
    st = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True)
    if not st.stdout.strip():
        print("لا تغييرات")
    else:
        subprocess.run(["git", "commit", "-m", f"Kaggle: SurahChain {SCN_PRESET} overnight pretrain"], check=False)
        subprocess.run(["git", "push", "origin", BRANCH], check=False)
        print("تم الرفع إلى GitHub")

## للنوم بأمان
1. **Save Version → Save & Run All** (ليس فقط Run التفاعلي)
2. تأكد Internet ON + GPU
3. في الجولة التالية: ضع `SCN_FRESH = False` ليكمل من آخر checkpoint
4. Kaggle ~30 ساعة GPU أسبوعياً وغالباً أطول استقراراً من Colab